# Experiment 49 v4 — Dependence-aware bootstrap and BH-FDR

This notebook re-tests the complete downstream audit after adding DLinear:

- 6 datasets × 5 backbones × 4 horizons = **120 registered conditions**;
- no forecaster or reranker is retrained;
- the loss differential is `Direct anchor MSE − Final anchor MSE`;
- the moving-block length is set prospectively to `ceil(horizon / anchor stride) + 1`, so it exceeds the overlap length;
- two-sided bootstrap p-values are adjusted jointly across all 120 conditions using Benjamini–Hochberg at `q=0.05`.

The notebook verifies every selected NPZ against the frozen absolute-metric table before inference. This prevents exploratory artifacts with similar filenames from entering the confirmatory family. Dynamic-block inference is recomputed for 112 conditions with saved paired series. The eight legacy Traffic PatchTST/TimeMixer conditions without saved paired arrays remain in the multiplicity family with the conservative assignment `p=1`; they therefore cannot increase the significance count.

In [ ]:
from pathlib import Path
from io import StringIO
import base64, json, math, re, zlib

import numpy as np
import pandas as pd

BASE = Path("/data/dataset/strong_forecaster")
OUT = BASE / "downstream_dynamic_block_bootstrap_fdr"
OUT.mkdir(parents=True, exist_ok=True)

HORIZONS = [96, 192, 336, 720]
DATASETS = ["ETTh1", "Weather", "Electricity", "Traffic", "Exchange", "Solar"]
BACKBONES = ["PatchTST", "iTransformer", "TimeMixer", "Seg-MoE", "DLinear"]
N_BOOT = 5000
Q = 0.05
SEED = 490027
METRIC_TOL = 5e-5

# Frozen 96-condition table from the submitted source bundle.
REFERENCE_B64 = "eNqtWl1vY8cNfe9vkQVyOMPhPKbINnlZoKgN9DFQXW1W6MYOvE6x6a/vITmSvZIfVvduHiKuLNyj4eHHIUc/7p53n/fPm7/u7v/zr8eH/ebnx6fD/x4fNj8envb3z+9v3/3y4fCw+7T5m///5Z+wftodHn75/f75+NEfvv7oD68++n7//HS4v3384+l+v8FfXv/7L+/u7j7y5u+75/uPd7d3m6Eb2ko3leKGla5lc8NbGtQF71QioTQa4TM/Hz4/4zvf7z79Y3//+N/9059vvHUOwsMfXll0VDcKqbCjaNdu+c6YxmBbBiLiR6mVhwVItarmIFXVErdylfm3IWMZTC/kT1Bu7K99aKubG9kO4VryLTHtbthQ4qtQbve/vn98l6TAFdrjOQL3hLuoFWvhSulDJI8Ev7378vv+6fDb/uFZ+u3+E8Jj/+833/waZtJirKO5MYqC6puyLVprz6OQ5meIdCzFSWaaNAkaWu2KxzuOWVOnvVFTdqDGpVtfCpTcaOvi5+lUqLTNjW5BUqnurta5twAcYjj0tTh3ePv94cv+aeaNmSGOYAymCig8WHsp4TIqrJk48PB1Mf0CMymqpbSgqDYydRytxGVmjEeZx0QzKgtxJkXcPf1glGY16kArwzIzO/IovoP5iRbiJENgma1HKOC/4RQBZoQr8fheI0xYGlLrGpzD3dPu4fOHx6ffTgyNkuULoVeMgyFCiYjoRoXLcsDUR1mOdCJJ8wR1EMC98jTr1TKFu/T5IR3LoWaRM6RqxNaICtfNgbIs1TYzV4oux5k8kTYKnkQLvvZN21pXynAbJatq4zb4m73n6YS3Ds9/nvUgRhyLhWGtuh+RWkr+JyRUH/OVr6p2b6ElW1zNsyUMDlzklvnBivPY0xgIwJVwyRhrkyhwqAkjUtkKWeRAafBkTQM1fSVcEofCZy1cRigdEgWjIxMCb3DX+BtaFi093+tOxXhOkzBIQjU0HjVqMOoWJ1hpqLjrwE7M9fAXDMlnQ76YJGP5RWCwlXVoR+LguJFGqek1ijcKOpZOo7aVR5u0MY3QEDDYPChRNzg6sBBLlEWguzJbhnbWvxjdN2QEjDyLIAYjq0vNlKvdrK5Gm8x1ksRVZHbknBbpESdKLToPjA5tuBZwkmd4PodRszajtFgUf+jcmgeFBMWXWgs4CSyasVk8OuJgw4UBHMtI9zQ8khbiXTY4blBmPbIBWaCbkLtKmQSsVtKQ9l0gJ48YFkqEjaKu+Tuur+J0nuVRTWFAYn0PzEll7xwqFQKu4ngQw0iWmiGD/tPT6NS+B+YxHVOA47VFlJKNUTIJBSUuDJz7qmN+uf+4e/h1f9b9vBeEb1+MqN5mmRVprIGZ1I06ormdDIqYVI4COo01OEmXILMHvzZiaoLgz6kyjTU4SRGcP/M6X0P/D9esL8YClNfdjaxT5NOLEYwgDvXFWA4yiTFBtXptBDHUORraNJajTFpQhqLgn4ykhfg4hl85Tp6hTFIIZIzXRtBixOHQaSxAOWtfaJYtxjoavq5ATfBa3jQbaJKzMMwuW1dvNF4bEQRj6JAXYxXQZAj6IubHkxEMca9SXoxVQE4So18Mf9LJSCDM+dm3prEA6LJH0cBMx18Z0SIlyMvXtUDHLColdNPJyCxqI3rkNNZinUocRRc8GZlLucjI17VIM58E0r2+NrLMQeTLi/HtWLePn3ZP5+OXSU2day5YeAvlNHrOQ65vwxCMDstxjhRZNFXGS+4aWq85blEuHyBXIO2X40zBMEaL3IRsMIhq3poN0xy0ZE54mG51OdBRJcAtEcgExVM3ZQuFZydVm7OCr2aud91XQ5aNXDmyoUn3jWwHKEuxiVoRagRiV4cuhUl+ivMdx4Fch0IWn0F0ijvjlufCUGJLcZKf0mv2T6QPnMV1WzEYc0ScYQjIwQ4Bx0txkh5MnRaDE9RhRZoKeSD44sS7YKcp5+iqSThxzqepAbIjhZCQUCNl28Y8h5dtmwPqglQ970bFVySpTAcpb6qPAd2mBtajGGbqK6AmTUjP5BySofLGtoYzpMRHJlMaNHSN+2YiNZG5eMH0y5sGR5aRpccwV6VR5Xr/XbakQpWieKJRdLRZ9lkUw0Oum8bJywuO9UZbKgj1WNaiVnDBkFa2qH6SR1NpacDRa8925AyitOTgYtBBjFrh1znxDXpLPV40Vsmr4I68FS9IuexBvMOXpl4UE24WdYSjXBOOwPnw4XB/cVPFI/orjB7F3FcKsZCE/3Id03q/SrlcICVp3uxq7pA5hj/kVdaRVrpNw3gN0NzmEjq4pdFD7SO/eCRbRbIJ91KvalUXWPPeSkzVphGkESd3o+RMD0PKklO9blaiWqPLw4gz0hxq0FxaFkKFwFiOMikySTEOo7aIBsbBMhpqzw1oq/jUcqQp9pBAQ6YRCzlCaZeZwFYzo2TQCs9NhqovUfOqIDdGBH2cJdgvFEsavS4501nHchf1adS4UEDLsNkSS96UwfALrTVYx3vGllEOb8W1C/plTmmCKa1Og67apF5izZRypRego/eeayOQEwdDhMROULxErQObtySCpt7SYI5bQEInCRKlqMVG1w1agnbZuyCPcycMwZyyQmo9tmSaxdDn4LYWbhJXrGREIiRCdrDSyLnX3Rx5jVYpZS3e8Qq/ysyA1hAONyEWc2qAHqiZ3zB0tT9nyvU+f+yA8ltm/ZLINLTr0afBV91f/HO/e/64vxiz6ogmxXHJyVv/TUXG5+jQ9WFo/ebr4guQ05ZvRj1i3gqUKCTboKmtLK9IxFwkLMSZQqPqlLd+yeq/G+icEgqDQgrePsZikCngkV1jZi6p+nyFMM9LXa+MnOssV/LX4Xw1XdVGnGt01AWfFjERaMvBy3LJjYn1qm3iGc5x/MUUEtRYQ08H0PBp1HKMy1jAkG9jLEc6klMK5VSozjTGEfwhToJ0mvdkUEq9LEeaDLFJ/gyCya9uy9ZzMyYfTKaUrRrdsdYFSOdDlhbKDFLyhYXfA/gGKTexcUPg92Oox9cFw8WIBXGsUWgIU6lEqmLyyasNDEXhPYyrpIuBJk3qrSGnazWPvO67o7ygpbx79H4vy4EmSyBFbGYNKkzBUFATCG9I/v5L0C+vBXrjUqpLsWwUijTCfFqINUUS2OK5LGlrgE7bippuZERXRQXHJFzmFJBObNCIvAZp8oRBUTJ5yJeYFc5STYkJR86mhPP2NViTqtZzIgBnPnTI1vxiOK4P6+xHYNO+lan/A1pl7ok="
reference96 = pd.read_csv(StringIO(zlib.decompress(base64.b64decode(REFERENCE_B64)).decode("utf-8")))

print("Output:", OUT)
print("Frozen original conditions:", len(reference96))
display(reference96.head())

## 1. Register the 120-condition family

The original 96 conditions use the frozen paper metrics. The 24 DLinear conditions are read from the six completed `summary.csv` files. All hypotheses are registered before any adjusted decision is computed.

In [ ]:
DLINEAR_ROOTS = {
    "Exchange": BASE / "exchange_dlinear_oof_historical_memory",
    "ETTh1": BASE / "etth1_dlinear_oof_historical_memory",
    "Weather": BASE / "weather_dlinear_oof_historical_memory",
    "Solar": BASE / "solar_dlinear_oof_historical_memory",
    "Electricity": BASE / "electricity_dlinear_oof_historical_memory",
    "Traffic": BASE / "traffic_dlinear_oof_historical_memory",
}

dlinear_rows = []
for dataset, root in DLINEAR_ROOTS.items():
    p = root / "summary.csv"
    if not p.is_file():
        raise FileNotFoundError(p)
    d = pd.read_csv(p)
    if len(d) != 4:
        raise RuntimeError(f"Expected four DLinear rows in {p}, found {len(d)}")
    for _, r in d.iterrows():
        dlinear_rows.append({
            "Dataset": dataset,
            "Backbone": "DLinear",
            "Horizon": int(r["Horizon"]),
            "DirectMSE_final": float(r["Direct_MSE"]),
            "FinalMSE_final": float(r["ShrinkAdaptive_MSE"]),
            "MSEGain_pct": float(r["MSEGain_pct"]),
        })

dlinear24 = pd.DataFrame(dlinear_rows)
family = pd.concat([
    reference96[["Dataset", "Backbone", "Horizon", "DirectMSE_final", "FinalMSE_final", "MSEGain_pct"]],
    dlinear24,
], ignore_index=True)
family["Backbone"] = family["Backbone"].replace({"SegMoE": "Seg-MoE"})
family = family.sort_values(["Dataset", "Backbone", "Horizon"]).reset_index(drop=True)

expected = pd.MultiIndex.from_product([DATASETS, BACKBONES, HORIZONS], names=["Dataset", "Backbone", "Horizon"]).to_frame(index=False)
check = expected.merge(family[["Dataset", "Backbone", "Horizon"]], how="left", indicator=True)
if len(family) != 120 or (check["_merge"] != "both").any():
    display(check[check["_merge"] != "both"])
    raise RuntimeError("The confirmatory family is not exactly 120 unique conditions.")

print("PASS: registered exactly 120 downstream conditions.")
display(family.groupby("Backbone").size().rename("Conditions"))

## 2. Resolve and verify paired anchor files

Candidate files are restricted to known final experiment roots. A file is accepted only when its anchor-level Direct and Final means reproduce the frozen table within tolerance. The final array key is selected by metric agreement rather than by a permissive filename rule.

In [ ]:
KNOWN_ROOTS = [
    BASE / "official_patchtst_plus_frozen_retrieval_h96",
    BASE / "official_patchtst_plus_frozen_retrieval_long_horizon",
    BASE / "weather_electricity_official_patchtst_plus_frozen_retrieval",
    BASE / "electricity_full321_frozen_transfer_standard",
    BASE / "itransformer_plus_frozen_historical_memory",
    BASE / "timemixer_plus_frozen_historical_memory",
    BASE / "solar_three_backbone_frozen_historical_memory",
    BASE / "traffic_three_backbone_frozen_historical_memory",
    BASE / "exchange_three_backbone_frozen_historical_memory",
    BASE / "etth1_full7_segmoe_oof_reranker",
    BASE / "weather_full21_segmoe_oof_reranker",
    BASE / "electricity_full321_segmoe_oof_reranker",
    BASE / "traffic_full862_segmoe_oof_reranker",
    BASE / "exchange_full8_segmoe_oof_reranker",
    BASE / "solar_full137_segmoe_oof_reranker",
] + list(DLINEAR_ROOTS.values())

def normalize_name(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

# Index all anchor-level audit files once. OOF and validation caches are
# excluded by the filename requirement.
ALL_ANCHOR_FILES = sorted({
    p for p in BASE.rglob("*.npz")
    if "anchormse" in normalize_name(str(p))
})
print("Indexed anchor-MSE files:", len(ALL_ANCHOR_FILES))

def candidate_files(dataset, backbone, horizon):
    ds = normalize_name(dataset)
    out = []
    bb = normalize_name(backbone)
    for p in ALL_ANCHOR_FILES:
        pn = normalize_name(p.name)
        full = normalize_name(str(p))
        # Seg-MoE files are named only H{horizon}_anchor_mse.npz; their
        # dataset identity is carried by the parent directory.
        if ds not in full or f"h{horizon}" not in full:
            continue
        if backbone == "DLinear" and "dlinear" not in full:
            continue
        if backbone == "Seg-MoE" and "segmoe" not in full:
            continue
        if backbone == "iTransformer" and "itransformer" not in full:
            continue
        if backbone == "TimeMixer" and "timemixer" not in full:
            continue
        if backbone == "PatchTST":
            if any(x in full for x in ["itransformerplus", "timemixerplus", "segmoe", "dlinear"]):
                continue
            if "threebackbone" in full and "patchtst" not in normalize_name(str(p.parent)):
                continue
        out.append(p)
    return out

def anchor_vector(a, n_anchor=None):
    a = np.asarray(a, dtype=np.float64)
    if a.ndim == 1:
        return a
    if n_anchor is not None and a.shape[0] == n_anchor:
        return a.reshape(n_anchor, -1).mean(axis=1)
    if n_anchor is not None and a.shape[-1] == n_anchor:
        return np.moveaxis(a, -1, 0).reshape(n_anchor, -1).mean(axis=1)
    return a.reshape(a.shape[0], -1).mean(axis=1)

def load_match(path, direct_ref, final_ref):
    with np.load(path) as z:
        keys = list(z.files)
        anchor_key = next((k for k in keys if normalize_name(k) in {"anchors", "anchor"}), None)
        n_anchor = len(z[anchor_key]) if anchor_key else None
        direct_keys = [k for k in keys if normalize_name(k) in {"direct", "directmse", "directanchormse"}]
        if not direct_keys:
            return None
        dk = min(direct_keys, key=lambda k: abs(anchor_vector(z[k], n_anchor).mean() - direct_ref))
        direct = anchor_vector(z[dk], n_anchor)
        # Final may be Direct (abstention), Scalar, RawAdaptive, or
        # ShrinkAdaptive. Metric agreement identifies the selected key.
        final_keys = [k for k in keys if normalize_name(k) not in {"anchors", "anchor"}]
        candidates = []
        for k in final_keys:
            try:
                v = anchor_vector(z[k], len(direct))
                if len(v) == len(direct):
                    candidates.append((abs(v.mean() - final_ref), k, v))
            except Exception:
                pass
        if not candidates:
            return None
        ferr, fk, final = min(candidates, key=lambda x: x[0])
        derr = abs(direct.mean() - direct_ref)
        if derr > METRIC_TOL or ferr > METRIC_TOL:
            return None
        anchors = np.asarray(z[anchor_key], dtype=np.int64) if anchor_key else np.arange(len(direct), dtype=np.int64)
        return {"Direct": direct, "Final": final, "Anchors": anchors, "DirectKey": dk, "FinalKey": fk,
                "DirectError": derr, "FinalError": ferr}

resolved = []
series = {}
unresolved = []
for _, row in family.iterrows():
    key = (row.Dataset, row.Backbone, int(row.Horizon))
    matches = []
    for p in candidate_files(*key):
        m = load_match(p, row.DirectMSE_final, row.FinalMSE_final)
        if m is not None:
            matches.append((p, m))
    if len(matches) == 0:
        candidates = candidate_files(*key)
        unresolved.append({"Dataset": key[0], "Backbone": key[1], "Horizon": key[2], "CandidateFiles": len(candidates)})
        print("UNRESOLVED", key, "candidate files:", len(candidates))
        for p in candidates[:12]:
            try:
                with np.load(p) as z:
                    print("  ", p, list(z.files))
            except Exception as e:
                print("  ", p, "LOAD ERROR:", e)
        continue
    matches.sort(key=lambda pm: (pm[1]["DirectError"] + pm[1]["FinalError"], len(str(pm[0])), str(pm[0])))
    p, m = matches[0]
    series[key] = m
    resolved.append({"Dataset": key[0], "Backbone": key[1], "Horizon": key[2], "Path": str(p),
                     "DirectKey": m["DirectKey"], "FinalKey": m["FinalKey"], "NAnchors": len(m["Direct"]),
                     "DirectMetricError": m["DirectError"], "FinalMetricError": m["FinalError"],
                     "EquivalentValidFiles": len(matches)})

manifest = pd.DataFrame(resolved)
manifest.to_csv(OUT / "00_verified_paired_manifest.csv", index=False)
unresolved_df = pd.DataFrame(unresolved)
allowed_conservative = {
    ("Traffic", "PatchTST", h) for h in HORIZONS
} | {
    ("Traffic", "TimeMixer", h) for h in HORIZONS
}
actual_unresolved = {
    tuple(x) for x in unresolved_df[["Dataset", "Backbone", "Horizon"]].itertuples(index=False, name=None)
} if len(unresolved_df) else set()
if actual_unresolved != allowed_conservative:
    display(manifest)
    display(unresolved_df)
    raise RuntimeError(
        f"Unexpected unresolved conditions: {sorted(actual_unresolved)}. "
        f"Expected only: {sorted(allowed_conservative)}"
    )
print("PASS: 112/120 paired conditions reproduce frozen metrics.")
print("Eight legacy Traffic conditions lack saved paired arrays and will receive conservative p=1.")
display(unresolved_df)
display(manifest.groupby("Backbone")["NAnchors"].agg(["count", "min", "median", "max"]))

## 3. Dependence-aware moving-block bootstrap

For adjacent anchors separated by `s` time steps, horizon-averaged target windows overlap for approximately `ceil(H/s)` anchors. We therefore use one block more than this overlap length. Circular rolling block means make the 5,000-replicate calculation fast without changing the forecasters.

In [ ]:
def bh_adjust(p):
    p = np.asarray(p, dtype=float)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adj = np.minimum.accumulate((ranked * m / np.arange(1, m + 1))[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.clip(adj, 0, 1)
    return out

def circular_block_means(x, L):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    ext = np.concatenate([x, x[:L-1]]) if L > 1 else x
    cs = np.concatenate([[0.0], np.cumsum(ext)])
    return (cs[L:L+n] - cs[:n]) / L

def fast_mbb(x, L, n_boot, seed):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    L = int(max(1, min(L, n)))
    block_means = circular_block_means(x, L)
    k = int(math.ceil(n / L))
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=np.float64)
    chunk = 500
    for lo in range(0, n_boot, chunk):
        hi = min(lo + chunk, n_boot)
        idx = rng.integers(0, n, size=(hi-lo, k))
        boot[lo:hi] = block_means[idx].mean(axis=1)
    # Percentile interval and finite-sample corrected two-sided p-value.
    left = (1 + np.count_nonzero(boot <= 0.0)) / (n_boot + 1)
    right = (1 + np.count_nonzero(boot >= 0.0)) / (n_boot + 1)
    return boot, min(1.0, 2.0 * min(left, right))

rows = []
for j, (_, r) in enumerate(family.iterrows()):
    key = (r.Dataset, r.Backbone, int(r.Horizon))
    if key not in series:
        # Conservative inclusion in the 120-test family: it cannot be called
        # significant, but its registered effect size remains visible.
        rows.append({
            "Dataset": r.Dataset, "Backbone": r.Backbone, "Horizon": int(r.Horizon),
            "NAnchors": np.nan, "AnchorStride": np.nan, "OverlapAnchors": np.nan,
            "DynamicBlockLength": np.nan, "EffectiveBlocks": np.nan,
            "MeanImprovement": float(r.DirectMSE_final - r.FinalMSE_final),
            "MSEGain_pct": float(r.MSEGain_pct), "CI_Low": np.nan, "CI_High": np.nan,
            "P_two_sided": 1.0, "InferenceStatus": "conservative_p1_missing_paired_series",
        })
        continue
    obj = series[key]
    anchors = obj["Anchors"]
    stride = int(max(1, round(np.median(np.diff(anchors))))) if len(anchors) > 1 else 1
    overlap = int(math.ceil(int(r.Horizon) / stride))
    L = min(len(anchors), overlap + 1)
    diff = obj["Direct"] - obj["Final"]
    boot, p = fast_mbb(diff, L, N_BOOT, SEED + j * 1009)
    rows.append({
        "Dataset": r.Dataset, "Backbone": r.Backbone, "Horizon": int(r.Horizon),
        "NAnchors": len(diff), "AnchorStride": stride, "OverlapAnchors": overlap,
        "DynamicBlockLength": L, "EffectiveBlocks": len(diff) / L,
        "MeanImprovement": diff.mean(), "MSEGain_pct": 100 * diff.mean() / obj["Direct"].mean(),
        "CI_Low": np.quantile(boot, 0.025), "CI_High": np.quantile(boot, 0.975),
        "P_two_sided": p, "InferenceStatus": "dynamic_block_bootstrap",
    })

result = pd.DataFrame(rows)
result["P_BH_120"] = bh_adjust(result["P_two_sided"])
result["FDR_Significant"] = result["P_BH_120"] <= Q
result["Direction"] = np.where(result["MeanImprovement"] > 0, "improve", np.where(result["MeanImprovement"] < 0, "degrade", "tie"))
result["FDR_Class"] = np.where(result["FDR_Significant"], result["Direction"], "not significant")
result.to_csv(OUT / "01_dynamic_block_bootstrap_bh_fdr_120.csv", index=False)

print("PASS: dependence-aware bootstrap and BH-FDR completed.")
display(result.head())

## 4. Confirmatory summaries and paper-ready outputs

In [ ]:
overall = pd.DataFrame([{
    "Conditions": len(result),
    "RawP_lt_0.05": int((result.P_two_sided < 0.05).sum()),
    "FDR_Significant": int(result.FDR_Significant.sum()),
    "FDR_Improvements": int(((result.FDR_Significant) & (result.Direction == "improve")).sum()),
    "FDR_Degradations": int(((result.FDR_Significant) & (result.Direction == "degrade")).sum()),
    "LowEffectiveBlockConditions": int((result.EffectiveBlocks.fillna(np.inf) < 4).sum()),
    "DynamicBootstrapConditions": int((result.InferenceStatus == "dynamic_block_bootstrap").sum()),
    "ConservativeP1Conditions": int((result.InferenceStatus != "dynamic_block_bootstrap").sum()),
}])

by_backbone = result.groupby("Backbone", as_index=False).agg(
    Conditions=("Horizon", "size"),
    MeanMSEGain_pct=("MSEGain_pct", "mean"),
    RawSignificant=("P_two_sided", lambda x: int((x < 0.05).sum())),
    FDRSignificant=("FDR_Significant", "sum"),
)
by_dataset = result.groupby("Dataset", as_index=False).agg(
    Conditions=("Horizon", "size"),
    MeanMSEGain_pct=("MSEGain_pct", "mean"),
    RawSignificant=("P_two_sided", lambda x: int((x < 0.05).sum())),
    FDRSignificant=("FDR_Significant", "sum"),
)

overall.to_csv(OUT / "02_overall_fdr_summary.csv", index=False)
by_backbone.to_csv(OUT / "03_fdr_by_backbone.csv", index=False)
by_dataset.to_csv(OUT / "04_fdr_by_dataset.csv", index=False)

paper_table = result[["Dataset", "Backbone", "Horizon", "MSEGain_pct", "DynamicBlockLength", "CI_Low", "CI_High", "P_two_sided", "P_BH_120", "FDR_Class"]]
paper_table.to_csv(OUT / "05_paper_ready_fdr_table.csv", index=False)
try:
    paper_table.to_latex(OUT / "05_paper_ready_fdr_table.tex", index=False, float_format="%.6f")
except Exception as e:
    print("LaTeX export warning:", e)

display(overall)
display(by_backbone)
display(by_dataset)

print("\nInterpretation rule:")
print("- Aggregate significance counts should use FDR_Significant / FDR_Class.")
print("- Conditions with EffectiveBlocks < 4 should be described as low-power, not as evidence of no effect.")
print("- Effect sizes remain primary; FDR decisions support only the aggregate significance claim.")

In [ ]:
required = [
    "00_verified_paired_manifest.csv",
    "01_dynamic_block_bootstrap_bh_fdr_120.csv",
    "02_overall_fdr_summary.csv",
    "03_fdr_by_backbone.csv",
    "04_fdr_by_dataset.csv",
    "05_paper_ready_fdr_table.csv",
]
missing = [name for name in required if not (OUT / name).is_file()]
if missing:
    raise RuntimeError(f"Missing outputs: {missing}")
print("PASS: all Experiment 49 outputs exist.")
for name in required:
    print(" -", OUT / name)